In [1]:
%matplotlib inline
import sys, os
sys.path.append("../src")
import numpy as np
import pandas as pd

from cleaning import load_all_clean
from eph_pass2 import EPH
import pipeline as pl

sectors_data = load_all_clean()
det2 = pl.detrend_all(sectors_data, EPH)
pl.save_detrended(det2, "../data/detrended_pass2")
print(len(det2), "sectors saved")

/home/shantanu/anaconda3/envs/toi700/lib/python3.11/site-packages/lightkurve/prf/__init__.py:7: UserWarning: Warning: the tpfmodel submodule is not available without oktopus installed, which requires a current version of autograd. See #1452 for details.
  warnings.warn(


37 sectors saved


In [2]:
#Run at start of every session

%matplotlib inline
import sys, os
sys.path.append("../src")
import numpy as np
import pandas as pd

from eph_pass2 import EPH
import pipeline as pl

det2 = pl.load_detrended("../data/detrended_pass2")
t_list, f_list, dy_list = pl.stack(det2)
t_all, f_all, dy_all = pl.concat_sorted(t_list, f_list, dy_list)
baseline = t_all.max() - t_all.min()
t_mid = 0.5 * (t_all.max() + t_all.min())
print(len(det2), "sectors |", len(t_all), "points | baseline:", round(baseline, 1))

37 sectors | 589399 points | baseline: 2663.0


In [3]:
#Bootstrap for BLS, about to be saved once

DUR0 = {"b": 2.193 / 24, "c": 1.0998 / 24, "d": 3.2742 / 24}   # best box widths from your BLS, in days

path = "../data/ephemeris_bootstrap_pass2.csv"
if os.path.exists(path):
    eph_boot = pd.read_csv(path)                      # already computed: just load it
else:
    rows = []
    for name, step in [("b", 1.0), ("c", 0.25), ("d", 1.0)]:
        P_s, T0_s = pl.bootstrap_ephemeris(t_list, f_list, dy_list,
                                           EPH[name]["P"], DUR0[name], t_mid, step_min=step)
        P_med, P_err = pl.summarize(P_s)
        T0_med, T0_err = pl.summarize(T0_s)
        rows.append({"planet": name, "step_min": step, "P_med": P_med, "P_err": P_err,
                     "T0_med": T0_med, "T0_err": T0_err})
        print(name, "done")
    eph_boot = pd.DataFrame(rows)
    eph_boot.to_csv(path, index=False)

print(eph_boot.to_string(index=False))

b done
c done
d done
planet  step_min     P_med    P_err      T0_med   T0_err
     b      1.00  9.977214 0.000022 2658.322832 0.001578
     c      0.25 16.051109 0.000012 2656.278942 0.000577
     d      1.00 37.423557 0.000098 2640.309705 0.002842
